# 05 — Hessam: Integration, Verdict Engine & Final Report

**CPU-only notebook — no GPU required, no network access required (beyond the one-time Drive
mount on Colab).** Unlike notebooks 01–04, this stage does no model training or inference; it
only fuses CSV/JSON outputs that already exist on disk, so it runs in well under a minute on a
plain CPU runtime, local or Colab.

Fuses every member's published outputs into one final JSON per invoice, runs the
user-configurable obligation-readiness **verdict engine** (`src/verdict_engine.py`) across the
whole batch under the Lenient / Default / Strict presets, writes the integration report, and
renders the two headline summary charts. **Run this last** — it consumes what notebooks 01–04
publish (or, run locally, what already lives in this repo's `outputs/` and `models/`).

This notebook **mirrors `scripts/build_final_json.py` exactly** — it imports and calls that
script's `main()` rather than reimplementing the fusion/verdict logic, so the notebook and the
CLI script can never silently drift apart.

| | |
|---|---|
| **Inputs** | All four members' outputs — Colab: Drive `inputs/` + `outputs/<member>/`; local: this repo's `data/processed/`, `outputs/predictions/`, `outputs/metrics/` (already the flat contract paths) |
| **Outputs** | `outputs/final_json/sample_invoice_outputs/*.json` (750 records), `outputs/reports/final_pipeline_report.md`, `outputs/figures/*.png` |
| **Expected runtime** | ~10–30 seconds (no training; CPU is plenty) |
| **Compute profile** | `local_cpu` (auto-detected — this stage never needs `colab_gpu`) |

### How results get back to the team
On Colab, the fused JSON + report + figures are published to Google Drive by
`colab_bootstrap.publish()`, into both `outputs/hessam/<kind>/` (latest) and
`runs/hessam/<UTC-timestamp>/<kind>/` (immutable archive). Run locally (as the orchestrator
does to verify), the same files simply land directly in this repo's `outputs/` — no publish step
needed, nothing to copy afterwards.


### Run order & prerequisites

This notebook auto-detects whether it's running on **Colab** or **locally** and adapts (see the
next two cells) — no manual editing needed either way.

**On Colab**, confirm `MyDrive/DL2_InvoiceAI/` already has:
```
code/{src,scripts,config}/                          # snapshot of the repo (config/ included!)
inputs/invoice_manifest.csv, inputs/annotations/*.csv
outputs/diana/predictions/stamp_signature_predictions.csv
outputs/diana/metrics/stamp_signature_metrics.json
outputs/jordan/predictions/region_predictions.csv
outputs/jordan/metrics/region_iou_metrics.json
outputs/damir/predictions/{ocr_outputs,parameter_presence_results,terms_extraction_results}.csv
outputs/damir/metrics/ocr_parameter_metrics.json
```
Missing pieces don't crash the notebook — the availability check a few cells down reports
exactly what's absent so you can integrate partial results and re-run later.

**Locally**, nothing to prepare — this repo already has every member's real output at the flat
contract paths (`outputs/predictions/*.csv`, `outputs/metrics/*.json`, `data/processed/invoice_manifest.csv`),
so the notebook reads them directly; just run the cells top to bottom from inside the repo
checkout.


In [ ]:
# --- No GPU needed for this notebook -------------------------------------------
# Integration is CSV/JSON fusion + rule evaluation, not model training or inference,
# so there is nothing here that benefits from a GPU. This cell is informational only
# (unlike 01-04's blocking GPU check) and never stops the notebook.
import sys

IN_COLAB = "google.colab" in sys.modules
try:
    import torch
    _cuda = torch.cuda.is_available()
except Exception:
    _cuda = False

print(f"Running in Colab: {IN_COLAB}")
print(f"CUDA available: {_cuda} (irrelevant here — this notebook only fuses existing outputs)")
print("This notebook is CPU-only by design. No GPU required.")


In [ ]:
# --- Bootstrap: mount Drive on Colab, or just locate the repo checkout locally -------
import os, shutil, json, time
from pathlib import Path

RUN_TS = time.strftime("%Y%m%dT%H%M%SZ", time.gmtime())
T0 = time.time()

if IN_COLAB:
    DRIVE_ROOT = "/content/drive/MyDrive/DL2_InvoiceAI"   # <-- change if your folder differs
    from google.colab import drive
    drive.mount("/content/drive")

    _bs = Path(DRIVE_ROOT) / "code" / "colab_bootstrap.py"
    assert _bs.exists(), (
        f"Missing {_bs}.\nUpload the repo's colab/colab_bootstrap.py into "
        f"{DRIVE_ROOT}/code/ and re-run this cell."
    )
    sys.path.insert(0, str(_bs.parent))
    import colab_bootstrap as CB

    root = CB.mount_drive(DRIVE_ROOT)
    cpaths = CB.setup_paths(root)
    CB.install_deps("pandas", "matplotlib")
    print("Drive root:", root)
else:
    # Local: walk up from the current working directory to find the repo checkout
    # (same marker find_repo_root() uses: requirements.txt + config/), then put it on
    # sys.path so `import src...` / `import scripts...` work exactly as they do for any
    # other script run from the repo root.
    _cwd = Path.cwd().resolve()
    REPO_ROOT = next(
        (c for c in [_cwd, *_cwd.parents] if (c / "requirements.txt").exists() and (c / "config").exists()),
        None,
    )
    assert REPO_ROOT is not None, (
        "Could not locate the repo checkout by walking up from "
        f"{_cwd}. Run this notebook from inside the invoice-image-processing repo."
    )
    if str(REPO_ROOT) not in sys.path:
        sys.path.insert(0, str(REPO_ROOT))
    print("Local repo root:", REPO_ROOT)


In [ ]:
# --- Resolve the flat contract paths this notebook's shared src/ modules expect --------
# `src.config.PATHS` is the single source of truth every shared module (results_store,
# final_json_builder, verdict_engine, streamlit_helpers, scripts/build_final_json) reads
# from. Locally it already points at this repo. On Colab there is no git checkout, so we
# stage a small local folder that mirrors the repo's flat contract layout (copying the
# member outputs out of Drive's per-member folders into it) and point PATHS at that
# instead -- every downstream cell then works identically in both environments.
import src.config as _cfgmod

if IN_COLAB:
    STAGE_ROOT = Path("/content/dl2_stage")
    for d in ["data/processed", "data/annotations", "config",
              "outputs/predictions", "outputs/metrics", "outputs/reports",
              "outputs/figures", "outputs/final_json/sample_invoice_outputs"]:
        (STAGE_ROOT / d).mkdir(parents=True, exist_ok=True)

    def _copy(src, dst):
        if Path(src).exists():
            shutil.copyfile(src, dst)
            return True
        return False

    _copy(cpaths.inputs / "invoice_manifest.csv", STAGE_ROOT / "data/processed/invoice_manifest.csv")

    for f in (cpaths.code / "config").glob("*.json") if (cpaths.code / "config").exists() else []:
        shutil.copyfile(f, STAGE_ROOT / "config" / f.name)

    ann_src = cpaths.inputs / "annotations"
    for f in (ann_src.glob("batch1_*.csv") if ann_src.exists() else []):
        shutil.copyfile(f, STAGE_ROOT / "data/annotations" / f.name)

    # member -> flat contract filename(s), per model_interface_contract.md
    PRED_MAP = {
        "diana": ["stamp_signature_predictions.csv"],
        "jordan": ["region_predictions.csv"],
        "damir": ["ocr_outputs.csv", "parameter_presence_results.csv", "terms_extraction_results.csv"],
    }
    METRIC_MAP = {
        "diana": ["stamp_signature_metrics.json"],
        "jordan": ["region_iou_metrics.json"],
        "damir": ["ocr_parameter_metrics.json"],
    }
    got, missing = [], []
    for member, files in PRED_MAP.items():
        for fn in files:
            ok = _copy(cpaths.outputs(member) / "predictions" / fn, STAGE_ROOT / "outputs/predictions" / fn)
            (got if ok else missing).append(f"{member}/predictions/{fn}")
    for member, files in METRIC_MAP.items():
        for fn in files:
            ok = _copy(cpaths.outputs(member) / "metrics" / fn, STAGE_ROOT / "outputs/metrics" / fn)
            (got if ok else missing).append(f"{member}/metrics/{fn}")

    print(f"staged {len(got)} upstream file(s) from Drive into {STAGE_ROOT}")
    if missing:
        print("MISSING (degrades gracefully -- see the availability check below):")
        for m in missing:
            print("  -", m)

    _cfgmod.PATHS = _cfgmod.ProjectPaths(repo_root=STAGE_ROOT)
else:
    print(f"Local run -- using this repo's own PATHS, nothing to stage.")

PATHS = _cfgmod.PATHS
print("\nrepo_root       :", PATHS.repo_root)
print("predictions_dir :", PATHS.predictions_dir)
print("metrics_dir     :", PATHS.metrics_dir)
print("final_json_dir  :", PATHS.final_json_dir)


In [ ]:
# --- What's actually on disk right now? -----------------------------------------------
# `results_store.availability()` is the same defensive check the Streamlit app uses, so the
# notebook and the app never disagree about what's ready to integrate.
from src import results_store as RS

avail = RS.availability()
print(json.dumps(avail, indent=2))

required = ["rolando_manifest", "diana_predictions", "jordan_predictions", "damir_ocr"]
missing_required = [k for k in required if not avail[k]]
assert not missing_required, (
    f"Missing required upstream output(s): {missing_required}. "
    "This notebook cannot fuse a final JSON without at least the manifest, stamp/signature "
    "predictions, region predictions, and OCR text -- see the prerequisites cell above."
)
print("\nAll required upstream outputs present -- ready to integrate.")


In [ ]:
# --- Run the real integration -- imports and calls scripts/build_final_json.py:main() --
# This is the SAME code the orchestrator runs from the command line (`python
# scripts/build_final_json.py`); the notebook does not reimplement any fusion/verdict logic,
# it just calls it, so the two can never silently drift apart. For every invoice in the
# manifest it fuses Diana + Jordan + Damir's outputs via `src.final_json_builder.build_final_json`,
# derives the reference/date/payment-terms verdict signals via `src.streamlit_helpers`, scores
# every invoice against the default policy AND all three presets via `src.verdict_engine`, and
# writes both the per-invoice JSON files and `outputs/reports/final_pipeline_report.md`.
from scripts import build_final_json as bfj

t0 = time.time()
bfj.main()
print(f"\nintegration finished in {time.time() - t0:.1f}s on CPU")


In [ ]:
# --- Load the fused records back + score every preset (for the charts below) -----------
# Recomputed here (not just parsed off bfj's printed summary) by calling the exact same
# shared functions build_final_json.py used internally -- signals_from_record, evaluate,
# preset_policies -- so this is still "mirror, don't reinvent", just producing an in-memory
# table the chart cells can plot directly.
from src.streamlit_helpers import signals_from_record
from src.verdict_engine import default_policy, evaluate, preset_policies

records = RS.load_final_records()
ocr_text = RS.invoice_ocr_text()
print(f"{len(records)} fused per-invoice JSON records loaded from {PATHS.final_json_dir}")

presets = preset_policies()
N = len(records)
ready_by_preset = {name: 0 for name in presets}
for rec in records:
    sig = signals_from_record(rec, ocr_text=ocr_text.get(rec.get("document_id")))
    for name, pol in presets.items():
        if evaluate(sig, pol).ready:
            ready_by_preset[name] += 1

print(f"\nObligation-readiness by policy ({N} invoices):")
for name in presets:
    r = ready_by_preset[name]
    print(f"  {name:8s}: {r:4d}/{N}  ({100 * r / N:.1f}%)")


In [ ]:
# --- Chart 1: obligation-readiness by verdict policy -----------------------------------
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

GOOD, WARNING, CRITICAL = "#0ca30c", "#eda100", "#d03b3b"
POLICY_ORDER = ["Lenient", "Default", "Strict"]
POLICY_COLOR = {"Lenient": GOOD, "Default": WARNING, "Strict": CRITICAL}

policies = [p for p in POLICY_ORDER if p in ready_by_preset] + \
           [p for p in ready_by_preset if p not in POLICY_ORDER]
vals = [ready_by_preset[p] for p in policies]
pct = [100 * v / N for v in vals]
colors = [POLICY_COLOR.get(p, "#4c78a8") for p in policies]

fig, ax = plt.subplots(figsize=(7, 4.6))
bars = ax.bar(policies, pct, color=colors, width=0.55, zorder=3)
for b, v, p in zip(bars, vals, pct):
    ax.annotate(f"{p:.1f}%\n({v}/{N})", xy=(b.get_x() + b.get_width() / 2, b.get_height()),
                xytext=(0, 6), textcoords="offset points", ha="center", va="bottom",
                fontsize=10.5, fontweight="bold")
ax.set_ylim(0, 115)
ax.set_ylabel("Invoices marked Ready (%)")
ax.set_title(f"Obligation-Readiness by Verdict Policy ({N} invoices)", fontsize=12.5, pad=12)
ax.grid(axis="y", color="#e1e0d9", linewidth=0.8, zorder=0)
ax.set_axisbelow(True)
for s in ["top", "right"]:
    ax.spines[s].set_visible(False)
fig.tight_layout()

FIG_DIR = PATHS.figures_dir
FIG_DIR.mkdir(parents=True, exist_ok=True)
fig.savefig(FIG_DIR / "readiness_by_policy.png", dpi=170, bbox_inches="tight")
plt.show()
print("saved ->", FIG_DIR / "readiness_by_policy.png")
print("\nStrict = 0/750 is honest, not a bug: this invoice corpus is clean, unsigned digital "
      "templates, and the Strict preset requires a visual mark. Diana's detector finds 0/750 "
      "stamps/signatures on THIS corpus while scoring stamp IoU 0.82 / signature IoU 0.81 on "
      "its own held-out (non-invoice) evaluation split -- a correct, fail-closed verdict given "
      "what the corpus actually contains, not a model failure.")


In [ ]:
# --- Chart 2: per-class detection quality (Diana + Jordan, from outputs/metrics/*.json) -
BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"

metrics = RS.load_metrics()
print("metrics files found:", list(metrics))

panels = []
if "stamp_signature" in metrics:
    m = metrics["stamp_signature"]
    classes = [c for c in ("stamp", "signature") if c in m]
    panels.append(("Diana -- stamp & signature\n(held-out SignverOD + StaVer split)",
                    classes, m, None))
if "region_iou" in metrics:
    m = metrics["region_iou"]
    pc = m.get("per_class", {})
    classes = list(pc)
    panels.append(("Jordan -- 5-class regions\n(OCR Dataset official test split)",
                    classes, pc, m.get("macro_mean_iou")))

if panels:
    fig, axes = plt.subplots(1, len(panels), figsize=(6.5 * len(panels), 4.6))
    if len(panels) == 1:
        axes = [axes]
    for ax, (title, classes, src, macro) in zip(axes, panels):
        x = range(len(classes))
        w = 0.26
        ax.bar([i - w for i in x], [src[c]["precision"] for c in classes], width=w, label="Precision", color=BLUE)
        ax.bar([i for i in x], [src[c]["recall"] for c in classes], width=w, label="Recall", color=ORANGE)
        ax.bar([i + w for i in x], [src[c]["mean_iou"] for c in classes], width=w, label="Mean IoU", color=AQUA)
        ax.set_xticks(list(x))
        ax.set_xticklabels(classes, rotation=0 if len(classes) <= 2 else 20, ha="center" if len(classes) <= 2 else "right")
        ax.set_ylim(0, 1.08)
        ax.set_title(title, fontsize=11, color="#333")
        ax.grid(axis="y", color="#e1e0d9", linewidth=0.8, zorder=0)
        ax.set_axisbelow(True)
        for s in ["top", "right"]:
            ax.spines[s].set_visible(False)
        if macro is not None:
            ax.axhline(macro, color="#898781", linestyle="--", linewidth=1.2)
            ax.annotate(f"macro mean IoU = {macro:.3f}", xy=(0.01, 0.99), xycoords="axes fraction",
                        ha="left", va="top", fontsize=8.5, color="#898781")
        ax.legend(frameon=False, fontsize=9)
    fig.suptitle("Detection Quality by Class (real held-out data)", fontsize=13, y=1.03)
    fig.tight_layout()
    fig.savefig(FIG_DIR / "metrics_per_class.png", dpi=170, bbox_inches="tight")
    plt.show()
    print("saved ->", FIG_DIR / "metrics_per_class.png")
else:
    print("No metrics JSONs found yet -- skipping this chart (degrades gracefully).")


In [ ]:
# --- The integration report, written by build_final_json.main() above ------------------
rep_path = PATHS.reports_dir / "final_pipeline_report.md"
print(rep_path.read_text(encoding="utf-8"))


In [ ]:
# --- Publish to Drive (Colab only) -------------------------------------------------------
if IN_COLAB:
    CB.publish("hessam", PATHS.final_json_dir, "predictions", paths=cpaths, run_timestamp=RUN_TS)
    CB.publish("hessam", rep_path, "logs", paths=cpaths, run_timestamp=RUN_TS)
    for f in FIG_DIR.glob("*.png"):
        CB.publish("hessam", f, "figures", paths=cpaths, run_timestamp=RUN_TS)
    print("Published to Drive ->", cpaths.outputs("hessam"))
    print("Archived to        ->", cpaths.run_dir("hessam", timestamp=RUN_TS))
    print("\nNext: copy outputs/hessam/predictions/sample_invoice_outputs/*.json into this "
          "repo's outputs/final_json/sample_invoice_outputs/, and the logs/metrics figures "
          "similarly, then `streamlit run app/streamlit_app.py` locally for the live demo.")
else:
    print("Local run -- nothing to publish. The fused JSON, report, and figures already live "
          "directly in this repo's outputs/ folder:")
    print(" ", PATHS.final_json_dir)
    print(" ", rep_path)
    print(" ", FIG_DIR)
    print("\nDemo it now:  streamlit run app/streamlit_app.py")

print(f"\ntotal notebook wall-clock: {time.time() - T0:.1f}s")


## Report log

Copy this into `presentation/member_reports/hessam_report_log.md` (or hand it to whoever owns
that file) as raw material for the group report and slide deck. Facts below are pre-filled from
this run's real output; adjust the two open prompts at the end.

1. **Stages integrated:** all four — Rolando's manifest (750 invoices), Diana's stamp/signature
   detections, Jordan's 5-class region detections, Damir's OCR text + parameter/terms
   extraction — fused into one JSON per invoice via `src.final_json_builder.build_final_json`.
   Nothing was missing at integration time; the availability check above is what would surface
   a gap if one existed on a future re-run.
2. **The end-to-end story:** 750/750 invoices got a fused final-JSON record. Under the
   **Default** policy (reference number present + parseable invoice date), **475/750 (63.3%)**
   are obligation-ready. **Lenient** (date-only) reaches **750/750 (100%)**. **Strict** (adds a
   required visual mark) is **0/750** — every rule fires, evaluated on real detector output, no
   hardcoded verdicts anywhere.
3. **Most important caveat for the audience:** Strict = 0% is a property of the *data*, not a
   model failure — this invoice corpus is unsigned digital templates, and Diana's stamp/signature
   detector is honestly reporting that (while scoring stamp IoU 0.82 / signature IoU 0.81 on its
   own held-out, non-invoice evaluation split). The verdict engine is fail-closed by design: an
   enabled rule with no signal counts as not-satisfied, so this 0% is trustworthy, not silently
   optimistic.
4. **What I'd do next with more compute or better ground truth:** _(fill in)_
5. **Anything the next stage / a figure for the slide:** `outputs/figures/readiness_by_policy.png`
   is the single chart that tells the whole obligation-readiness story at a glance.
